# SciPyWrapper-Style Distribution Wrapping Demo

Original QMCPy demo: [`QMCPy/demos/scipywrapper_dependence_custom/scipywrapper_demo.ipynb`](../../../QMCPy/demos/scipywrapper_dependence_custom/scipywrapper_demo.ipynb)

`QMC.jl` uses `DistributionsWrapper` as the direct analogue of QMCPy's `SciPyWrapper`. This translation also includes two custom true measures from `QMC.jl` that mirror the spirit of the Python notebook.


In [1]:
using QMC
using Distributions
using Statistics
using Printf


## Mixed Marginals with `DistributionsWrapper`

Wrap a discrete distribution with standard `Distributions.jl` marginals. The transformed samples should match the requested one-dimensional laws while preserving the low-discrepancy structure of the underlying point set.


In [2]:
dd = Lattice(3; seed=7)
tm = DistributionsWrapper(dd; marginals=[Normal(), Exponential(2.0), Beta(2, 5)])
x = transform(tm, gen_samples(dd, 4096))

summary_rows = [
    (name="Normal(0,1)", empirical_mean=mean(x[:, 1]), theoretical_mean=mean(Normal()), empirical_std=std(x[:, 1])),
    (name="Exponential(2)", empirical_mean=mean(x[:, 2]), theoretical_mean=mean(Exponential(2.0)), empirical_std=std(x[:, 2])),
    (name="Beta(2,5)", empirical_mean=mean(x[:, 3]), theoretical_mean=mean(Beta(2, 5)), empirical_std=std(x[:, 3])),
]
foreach(println, summary_rows)

@assert abs(summary_rows[1].empirical_mean - summary_rows[1].theoretical_mean) < 0.05
@assert abs(summary_rows[2].empirical_mean - summary_rows[2].theoretical_mean) < 0.05
@assert abs(summary_rows[3].empirical_mean - summary_rows[3].theoretical_mean) < 0.02


(name

 = "Normal(0,1)", empirical_mean = 0.0010604139539520888, theoretical_mean = 0.0, empirical_std = 1.0006474193648778)
(name = "Exponential(2)", empirical_mean = 2.0000703052425433, theoretical_mean = 2.0, empirical_std = 1.9996275204186211)
(name = "Beta(2,5)", empirical_mean = 0.28564712859909086, theoretical_mean = 0.2857142857142857, empirical_std = 0.1596651172533297)


## Custom True Measures Available in `QMC.jl`

The Python notebook also mixes in custom true measures. Two lightweight Julia examples are `ZeroInflatedExpUniform` and `UniformTriangle`.


In [3]:
zi_tm = ZeroInflatedExpUniform(Lattice(2; seed=7); p_zero=0.3, rate=2.0)
zi_x = transform(zi_tm, gen_samples(zi_tm.dd, 4096))
zero_fraction = mean(vec(zi_x) .== 0.0)
@printf("ZeroInflatedExpUniform: zero fraction = %.4f, sample mean = %.4f
", zero_fraction, mean(zi_x))

tri_tm = UniformTriangle(Lattice(2; seed=7))
tri_x = transform(tri_tm, gen_samples(tri_tm.dd, 4096))
@printf("UniformTriangle: mean(x₁) = %.4f, mean(x₂) = %.4f, constraint satisfied = %s
", mean(tri_x[:, 1]), mean(tri_x[:, 2]), all(tri_x[:, 1] .>= tri_x[:, 2]))

@assert abs(zero_fraction - 0.3) < 0.02
@assert all(tri_x[:, 1] .>= tri_x[:, 2])


ZeroInflatedExpUniform: zero fraction = 0.3000, sample mean = 0.3497
UniformTriangle: mean(x₁) = 0.6668, mean(x₂) = 0.3334, constraint satisfied = true
